# Day 32: Graph RAG Prototype

## Core Theory (Just-in-Time)

Welcome to Day 32! Today we are diving into **Graph RAG (Retrieval-Augmented Generation)**. 
Traditional RAG relies on semantic similarity using vector embeddings. While great for finding relevant text, it struggles with complex, multi-hop queries where relationships between entities are key (e.g., "Which companies acquired startups funded by Sequoia in 2023?").

Graph RAG solves this by extracting **Entities** (nodes) and **Relationships** (edges) from unstructured text to build a **Knowledge Graph**. When a user asks a question, we query the graph to find connected entities and use the structured context to augment the LLM's response.

### Why Graph RAG?
- **High Precision:** Reduces hallucinations by grounding answers in explicit, structured relationships.
- **Complex Reasoning:** Enables multi-hop reasoning across disparate documents.
- **Explainability:** You can trace exactly which relationships were used to generate an answer.

### How it Works
1. **Extraction:** Use an LLM to parse text and extract entities (e.g., Person, Organization) and their relationships (e.g., "WORKS_FOR", "FOUNDED").
2. **Construction:** Store these nodes and edges in a graph structure (or graph database).
3. **Retrieval:** Traverse the graph to find relevant subgraphs based on the user's query.
4. **Generation:** Pass the retrieved subgraph to the LLM to generate the final answer.

### AI Security Implications
- **PII Protection:** Entities extracted might contain sensitive data (e.g., personal emails, phone numbers). Ensure PII is redacted *before* passing text to the LLM for extraction to prevent data leakage.
- **Prompt Injection:** Because we parse arbitrary user text, malicious inputs could manipulate the extraction (e.g., injecting false relationships). Use strict system prompts and structured output parsing.
- **Fallbacks:** External LLM calls can fail. Implement `try...except` blocks with sensible fallbacks (like returning an empty KnowledgeGraph object) to prevent system crashes.


## Code Implementation

We will explore Graph RAG extraction and querying through three tiers: Basic, Medium, and Advanced.


### Basic: Core Extraction Concept
This isolates the core concept of extraction using LangChain's structured output. We use `langchain-openai` directly.


In [1]:
import os
from pydantic import BaseModel, Field
from typing import List
from langchain_openai import ChatOpenAI

class BasicEntity(BaseModel):
    name: str = Field(..., description="Entity name")
    type: str = Field(..., description="Entity type")

class BasicGraph(BaseModel):
    entities: List[BasicEntity]

def basic_extract(text: str) -> BasicGraph:
    # For local validation, we use a try-except to handle missing dummy keys
    try:
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
        structured = llm.with_structured_output(BasicGraph)
        return structured.invoke(text)
    except Exception as e:
        print(f"Fallback: Returning empty graph due to {e}")
        return BasicGraph(entities=[])

if __name__ == "__main__":
    text = "Sam Altman founded OpenAI in San Francisco."
    print(basic_extract(text))


Fallback: Returning empty graph due to Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}
entities=[]


### Medium: Clean OOP and State Management
Here we introduce a simple in-memory graph using NetworkX. It demonstrates clean object interactions.


In [2]:
import networkx as nx
from typing import List
from pydantic import BaseModel, Field

class Entity(BaseModel):
    name: str
    type: str

class Relationship(BaseModel):
    source: str
    target: str
    relation: str

class KnowledgeGraph(BaseModel):
    entities: List[Entity] = Field(default_factory=list)
    relationships: List[Relationship] = Field(default_factory=list)

class MediumGraphRAG:
    def __init__(self):
        self.graph = nx.DiGraph()
        
    def ingest_graph(self, kg: KnowledgeGraph) -> None:
        for entity in kg.entities:
            self.graph.add_node(entity.name, type=entity.type)
        for rel in kg.relationships:
            self.graph.add_edge(rel.source, rel.target, relation=rel.relation)
            
    def query_entity(self, entity_name: str) -> str:
        if not self.graph.has_node(entity_name):
            return "Entity not found."
        subgraph = nx.ego_graph(self.graph, entity_name, radius=1)
        return "\n".join(f"{u} -[{data.get('relation')}]-> {v}" for u, v, data in subgraph.edges(data=True))

if __name__ == "__main__":
    kg = KnowledgeGraph(
        entities=[Entity(name="Sam", type="PERSON"), Entity(name="OpenAI", type="ORG")],
        relationships=[Relationship(source="Sam", target="OpenAI", relation="FOUNDED")]
    )
    rag = MediumGraphRAG()
    rag.ingest_graph(kg)
    print(rag.query_entity("Sam"))


Sam -[FOUNDED]-> OpenAI


### Advanced: Production-Ready Graph RAG
This implementation features strict type hinting, robust error handling, PII redaction setup, and exact import syntax.


In [3]:
import os
import logging
from typing import List, Dict, Any, Optional
from pydantic import BaseModel, Field, ValidationError
import networkx as nx
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class AdvEntity(BaseModel):
    """Represents a recognized entity with strict typing."""
    name: str = Field(..., description="The properly capitalized entity name.")
    type: str = Field(..., description="Entity category, e.g., PERSON, ORGANIZATION.")
    properties: Dict[str, Any] = Field(default_factory=dict, description="Additional metadata.")

class AdvRelationship(BaseModel):
    """Defines a directed edge between two entities."""
    source: str = Field(..., description="Source entity name.")
    target: str = Field(..., description="Target entity name.")
    relation_type: str = Field(..., description="Standardized relation, e.g., FOUNDED, ACQUIRED.")
    properties: Dict[str, Any] = Field(default_factory=dict, description="Additional edge metadata.")

class AdvKnowledgeGraph(BaseModel):
    """The overall Knowledge Graph container."""
    entities: List[AdvEntity] = Field(default_factory=list)
    relationships: List[AdvRelationship] = Field(default_factory=list)

class ProductionGraphExtractor:
    """Handles extraction with error boundaries and security fallbacks."""
    
    def __init__(self, model_name: str = "gpt-4o-mini"):
        # Securely fetch API key without hardcoding
        api_key = os.environ.get("OPENAI_API_KEY")
        if not api_key:
            logger.warning("OPENAI_API_KEY is missing; extraction will fail gracefully.")
        self.llm = ChatOpenAI(model=model_name, temperature=0.0, api_key=api_key)
        self.prompt = ChatPromptTemplate.from_messages([
            ("system", "Extract entities and relationships into a precise knowledge graph. Sanitize any obvious PII."),
            ("human", "{text}")
        ])
        self.chain = self.prompt | self.llm.with_structured_output(AdvKnowledgeGraph)

    def extract(self, text: str) -> AdvKnowledgeGraph:
        """Extracts the KG with safe error handling."""
        try:
            return self.chain.invoke({"text": text})
        except Exception as e:
            logger.error(f"Extraction failed: {e}. Returning empty graph.")
            return AdvKnowledgeGraph()

if __name__ == "__main__":
    extractor = ProductionGraphExtractor()
    kg = extractor.extract("Elon Musk acquired Twitter (now X) for $44 billion.")
    # Pydantic v2 model_dump_json
    logger.info(f"Extracted: {kg.model_dump_json(indent=2)}")


INFO:httpx2:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 401 Unauthorized"


ERROR:__main__:Extraction failed: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-dummy. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}. Returning empty graph.


INFO:__main__:Extracted: {
  "entities": [],
  "relationships": []
}


## Practical Lab / Homework

**Your Task:**
1. Extend the `extract_knowledge_graph` function to also extract **Properties** (e.g., date founded, investment amount) and attach them to the nodes or edges.
2. Update the Pydantic models to support a `properties` dictionary field.
3. Write a small script that takes a user query (e.g., "Who invested in the company Sam Altman founded?"), uses an LLM to identify the target entity in the query ("Sam Altman"), retrieves the subgraph context, and answers the question using the retrieved context.

*Hint: Use a standard LCEL chain for the final generation step, passing the `get_context_for_entity` output as context.*

*Please record a brief async video walkthrough of your design decisions when completed.*


In [4]:
# LAB WORK: Implement your solution here
import os
from typing import List
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# 1. Update Pydantic Models (Entity and Relationship with properties)
class LabEntity(BaseModel):
    name: str = Field(..., description="Name of entity")
    type: str = Field(..., description="Type of entity")
    properties: dict = Field(default_factory=dict)

class LabRelationship(BaseModel):
    source: str = Field(..., description="Source entity name")
    target: str = Field(..., description="Target entity name")
    relation: str = Field(..., description="Relation type")
    properties: dict = Field(default_factory=dict)

class LabGraph(BaseModel):
    entities: List[LabEntity] = Field(default_factory=list)
    relationships: List[LabRelationship] = Field(default_factory=list)

# 2. Update Extraction Logic
def lab_extraction(text: str) -> LabGraph:
    try:
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
        chain = llm.with_structured_output(LabGraph)
        return chain.invoke(text)
    except Exception as e:
        print(f"Fallback: {e}")
        return LabGraph()

# 3. Create the end-to-end QA chain


## Common Pitfalls

When building Graph RAG in production, look out for:

1. **Entity Resolution / Deduplication:** The LLM might extract "Apple", "Apple Inc.", and "Apple Computer" as three different nodes. You need a normalization or entity resolution step (often using embeddings) to merge identical entities.
2. **Schema Drift:** If you don't constrain the LLM (like we did with Pydantic), it will invent hundreds of slightly different relation types (e.g., `WORKS_AT`, `EMPLOYED_BY`, `IS_EMPLOYEE_OF`). Strict typing and ENUMs are essential.
3. **Graph Traversal Explosion:** Doing deep traversal (depth > 2) on a densely connected graph (like highly referenced entities) can pull in too much context, confusing the LLM and blowing up context windows.
4. **Extraction Latency:** Extracting graphs dynamically for every document at query time is too slow. Extraction must be done asynchronously during the ingestion pipeline.


## Reference Links
1. [LangChain Graph QA Documentation](https://python.langchain.com/docs/use_cases/graph/)
2. [NetworkX Documentation](https://networkx.org/documentation/stable/)
3. [OpenAI Structured Outputs](https://platform.openai.com/docs/guides/structured-outputs)
